# Mid-Block Expansion Evaluation on GSM8K

评测 **Dual Cache + Mid-Block Chain Expansion** 在 GSM8K 上的效果。

**三组实验：**
1. **Baseline**: 原版 dual cache（`mid_trigger_ratio=0`，不触发扩展）
2. **Expand (no rewarm)**: 开启扩展，关闭 re-warm（`rewarm_on_expand=False`）
3. **Expand (rewarm)**: 开启扩展，开启 re-warm（`rewarm_on_expand=True`）

**核心指标：**
- GSM8K 准确率（100 题）
- 每步解码 token 数量分布
- 总 NFE 对比

## 1. 环境设置

In [ ]:
import os
import torch
import gc

# ========== 修改这里选择 GPU ==========
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# 环境变量
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

# 切换到 llada 目录
os.chdir('llada')

# 清理 GPU 缓存
torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 加载模型

In [ ]:
from transformers import AutoTokenizer
from model.modeling_llada import LLaDAModelLM

device = 'cuda'
model_name = 'GSAI-ML/LLaDA-8B-Instruct'

print(f"Loading model: {model_name}")
model = LLaDAModelLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Model loaded!")

## 3. 加载 GSM8K 数据集（100 题）

In [ ]:
from datasets import load_dataset
import re

# 加载 GSM8K 测试集
ds = load_dataset("openai/gsm8k", "main", split="test")

# 取前 100 个样本
LIMIT = 100
ds = ds.select(range(LIMIT))

print(f"Loaded {len(ds)} GSM8K samples")
print(f"Example question: {ds[0]['question'][:120]}...")
print(f"Example answer:   {ds[0]['answer'][-60:]}...")


def extract_ref_answer(answer_text: str) -> str:
    """从 GSM8K 参考答案中提取 #### 后面的数字。"""
    match = re.search(r'####\s*([+-]?[\d,]+\.?\d*)', answer_text)
    return match.group(1).replace(',', '') if match else None


def extract_gen_answer(text: str) -> str:
    """从模型生成文本中提取最后一个数字作为答案。"""
    # 优先找 #### 模式
    match = re.search(r'####\s*([+-]?[\d,]+\.?\d*)', text)
    if match:
        return match.group(1).replace(',', '')
    # boxed 模式
    match = re.search(r'\\boxed\{([+-]?[\d,]+\.?\d*)\}', text)
    if match:
        return match.group(1).replace(',', '')
    # 回退：最后一个数字
    numbers = re.findall(r'[+-]?[\d,]+\.?\d*', text)
    if numbers:
        return numbers[-1].replace(',', '')
    return None


# 预处理参考答案
ref_answers = [extract_ref_answer(d['answer']) for d in ds]
print(f"\nFirst 5 reference answers: {ref_answers[:5]}")

# 准备 prompts（instruct 格式）
prompts = []
for d in ds:
    m = [{"role": "user", "content": d['question']}]
    prompt_text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)
    prompts.append(input_ids)

print(f"Prepared {len(prompts)} prompts (avg len: {sum(p.shape[1] for p in prompts)/len(prompts):.0f} tokens)")

## 4. 定义评测函数

In [ ]:
from generate import generate_with_dual_cache_expand
from tqdm.auto import tqdm
import time
import json

# ========== 生成超参 ==========
GEN_LENGTH   = 256
STEPS        = 256
BLOCK_LENGTH = 32
THRESHOLD    = 0.9       # token transfer threshold
MASK_ID      = 126336


def run_eval(config_name, mid_trigger_ratio, rewarm_on_expand, prompts, ref_answers):
    """
    跑一组配置，返回结果字典。
    
    config_name: 配置名（用于日志）
    mid_trigger_ratio: 0.0 = baseline（不扩展）, 0.5 = 中点扩展
    rewarm_on_expand: 扩展时是否 re-warm
    """
    results = {
        'config': config_name,
        'mid_trigger_ratio': mid_trigger_ratio,
        'rewarm_on_expand': rewarm_on_expand,
        'correct': 0,
        'total': len(prompts),
        'nfe_list': [],
        'all_step_records': [],    # list of per-sample step records
        'answers': [],
    }
    
    start_time = time.time()
    
    for idx in tqdm(range(len(prompts)), desc=config_name):
        input_ids = prompts[idx]
        prompt_len = input_ids.shape[1]
        
        with torch.inference_mode():
            x, nfe, step_records = generate_with_dual_cache_expand(
                model, input_ids,
                steps=STEPS,
                gen_length=GEN_LENGTH,
                block_length=BLOCK_LENGTH,
                temperature=0.,
                threshold=THRESHOLD,
                mid_trigger_ratio=mid_trigger_ratio,
                rewarm_on_expand=rewarm_on_expand,
                record_steps=True,
            )
        
        # 解码生成结果
        gen_text = tokenizer.decode(x[0, prompt_len:], skip_special_tokens=True)
        gen_answer = extract_gen_answer(gen_text)
        ref_answer = ref_answers[idx]
        
        is_correct = (gen_answer == ref_answer) if gen_answer and ref_answer else False
        
        results['correct'] += int(is_correct)
        results['nfe_list'].append(nfe)
        results['all_step_records'].append(step_records)
        results['answers'].append({
            'idx': idx,
            'gen_text': gen_text[:300],
            'gen_answer': gen_answer,
            'ref_answer': ref_answer,
            'correct': is_correct,
            'nfe': nfe,
            'num_steps': len(step_records),
        })
    
    elapsed = time.time() - start_time
    results['elapsed_sec'] = elapsed
    results['accuracy'] = results['correct'] / results['total']
    results['avg_nfe'] = sum(results['nfe_list']) / len(results['nfe_list'])
    
    print(f"\n{'='*50}")
    print(f"Config: {config_name}")
    print(f"  Accuracy: {results['correct']}/{results['total']} = {results['accuracy']*100:.1f}%")
    print(f"  Avg NFE:  {results['avg_nfe']:.1f}")
    print(f"  Time:     {elapsed:.1f}s")
    print(f"{'='*50}")
    
    return results


print("Eval function ready.")

## 5. 运行三组评测

依次运行三个配置。如需在 3 个 GPU 上并行运行，可以复制本 notebook 三份，分别设置不同的 `CUDA_VISIBLE_DEVICES` 和只运行对应的 config cell。

In [ ]:
# Config 1: Baseline (原版 dual cache，不触发扩展)
res_baseline = run_eval(
    config_name="baseline",
    mid_trigger_ratio=0.0,     # 永远不触发扩展
    rewarm_on_expand=True,     # 无影响（不会触发）
    prompts=prompts,
    ref_answers=ref_answers,
)

In [ ]:
# Config 2: Expand + no rewarm
res_expand_no_rewarm = run_eval(
    config_name="expand_no_rewarm",
    mid_trigger_ratio=0.5,
    rewarm_on_expand=False,
    prompts=prompts,
    ref_answers=ref_answers,
)

In [ ]:
# Config 3: Expand + rewarm
res_expand_rewarm = run_eval(
    config_name="expand_rewarm",
    mid_trigger_ratio=0.5,
    rewarm_on_expand=True,
    prompts=prompts,
    ref_answers=ref_answers,
)

## 6. 保存结果

保存为 JSON 方便跨机器汇总。

In [ ]:
import datetime

os.makedirs('../eval_results', exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

all_configs = [res_baseline, res_expand_no_rewarm, res_expand_rewarm]

for res in all_configs:
    fname = f"../eval_results/gsm8k_{res['config']}_{timestamp}.json"
    # step_records 含 tensor 相关数据，只保留可序列化的部分
    save_data = {
        'config': res['config'],
        'mid_trigger_ratio': res['mid_trigger_ratio'],
        'rewarm_on_expand': res['rewarm_on_expand'],
        'accuracy': res['accuracy'],
        'correct': res['correct'],
        'total': res['total'],
        'avg_nfe': res['avg_nfe'],
        'elapsed_sec': res['elapsed_sec'],
        'nfe_list': res['nfe_list'],
        'answers': res['answers'],
        'all_step_records': res['all_step_records'],
        'gen_params': {
            'gen_length': GEN_LENGTH,
            'steps': STEPS,
            'block_length': BLOCK_LENGTH,
            'threshold': THRESHOLD,
        },
    }
    with open(fname, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    print(f"Saved: {fname}")

print("All results saved.")

## 7. 准确率对比

In [ ]:
import pandas as pd
import numpy as np

# 汇总表格
summary_rows = []
for res in all_configs:
    # 统计扩展次数
    expand_steps = sum(
        1 for records in res['all_step_records']
        for r in records if r['type'] == 'expand'
    )
    summary_rows.append({
        'Config': res['config'],
        'Accuracy': f"{res['accuracy']*100:.1f}%",
        'Correct': f"{res['correct']}/{res['total']}",
        'Avg NFE': f"{res['avg_nfe']:.1f}",
        'Total Expand Steps': expand_steps,
        'Time (s)': f"{res['elapsed_sec']:.1f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

## 8. 每步解码 Token 数量分布（核心图表）

对每个 config，统计所有样本在每个 global_step 上解码了多少 token，绘制均值 + 标准差带。

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

colors = {'baseline': '#2196F3', 'expand_no_rewarm': '#FF9800', 'expand_rewarm': '#4CAF50'}
config_labels = {'baseline': 'Baseline (dual cache)', 'expand_no_rewarm': 'Expand (no rewarm)', 'expand_rewarm': 'Expand (rewarm)'}

for ax_idx, res in enumerate(all_configs):
    cname = res['config']
    ax = axes[ax_idx]
    
    # 收集每个 global_step 的 transferred 数量
    step_transferred = defaultdict(list)
    for sample_records in res['all_step_records']:
        for r in sample_records:
            step_transferred[r['global_step']].append(r['transferred'])
    
    max_step = max(step_transferred.keys()) if step_transferred else 0
    steps_range = list(range(max_step + 1))
    means = [np.mean(step_transferred[s]) if s in step_transferred else 0 for s in steps_range]
    stds  = [np.std(step_transferred[s])  if s in step_transferred else 0 for s in steps_range]
    
    ax.bar(steps_range, means, color=colors[cname], alpha=0.7, label='Mean transferred')
    ax.errorbar(steps_range, means, yerr=stds, fmt='none', ecolor='gray', alpha=0.4, capsize=1)
    ax.set_xlabel('Global Step')
    ax.set_title(f"{config_labels[cname]}\nAcc={res['accuracy']*100:.1f}%, Avg NFE={res['avg_nfe']:.1f}")
    ax.set_xlim(-0.5, min(max_step + 0.5, 80))  # 限制 x 轴范围便于查看

axes[0].set_ylabel('Tokens Transferred')
fig.suptitle('Per-Step Decoded Token Count (GSM8K, 100 samples)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../eval_results/per_step_tokens.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eval_results/per_step_tokens.png")

## 9. 三组 Config 叠加对比（均值线图）

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for res in all_configs:
    cname = res['config']
    
    step_transferred = defaultdict(list)
    for sample_records in res['all_step_records']:
        for r in sample_records:
            step_transferred[r['global_step']].append(r['transferred'])
    
    max_step = max(step_transferred.keys()) if step_transferred else 0
    steps_range = list(range(max_step + 1))
    means = [np.mean(step_transferred[s]) if s in step_transferred else 0 for s in steps_range]
    stds  = [np.std(step_transferred[s])  if s in step_transferred else 0 for s in steps_range]
    
    ax.plot(steps_range, means, label=config_labels[cname], color=colors[cname], linewidth=1.5, alpha=0.9)
    ax.fill_between(steps_range,
                     [m - s for m, s in zip(means, stds)],
                     [m + s for m, s in zip(means, stds)],
                     color=colors[cname], alpha=0.15)

ax.set_xlabel('Global Step', fontsize=12)
ax.set_ylabel('Tokens Transferred (mean +/- std)', fontsize=12)
ax.set_title('Per-Step Token Decode Comparison (GSM8K, 100 samples)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, 80)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../eval_results/per_step_tokens_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eval_results/per_step_tokens_overlay.png")

## 10. Step 类型分布（warm / refine / expand）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

type_colors = {'warm': '#2196F3', 'refine': '#90CAF9', 'expand': '#FF5722'}

for ax_idx, res in enumerate(all_configs):
    cname = res['config']
    ax = axes[ax_idx]
    
    # 统计每种类型的 step 数量和总 transferred
    type_counts = defaultdict(int)
    type_transferred = defaultdict(int)
    for sample_records in res['all_step_records']:
        for r in sample_records:
            type_counts[r['type']] += 1
            type_transferred[r['type']] += r['transferred']
    
    types = ['warm', 'refine', 'expand']
    counts = [type_counts.get(t, 0) for t in types]
    transferred = [type_transferred.get(t, 0) for t in types]
    
    x_pos = np.arange(len(types))
    bars = ax.bar(x_pos, counts, color=[type_colors[t] for t in types], alpha=0.8)
    
    # 在柱子上标注总 transferred
    for bar, t_count in zip(bars, transferred):
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'tok={t_count}', ha='center', va='bottom', fontsize=9)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(types)
    ax.set_title(f"{config_labels[cname]}")
    ax.set_ylabel('Step Count' if ax_idx == 0 else '')

fig.suptitle('Step Type Distribution (total across all 100 samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../eval_results/step_type_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. NFE 分布对比

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

positions = np.arange(len(all_configs))
nfe_means = [np.mean(res['nfe_list']) for res in all_configs]
nfe_stds  = [np.std(res['nfe_list'])  for res in all_configs]
labels = [config_labels[res['config']] for res in all_configs]
bar_colors = [colors[res['config']] for res in all_configs]

bars = ax.bar(positions, nfe_means, yerr=nfe_stds, color=bar_colors, alpha=0.8,
              capsize=5, edgecolor='gray')

for bar, mean, acc in zip(bars, nfe_means, [r['accuracy'] for r in all_configs]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'NFE={mean:.1f}\nAcc={acc*100:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('NFE (Number of Forward Evaluations)', fontsize=12)
ax.set_title('NFE Comparison (GSM8K, 100 samples)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../eval_results/nfe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. 逐样本正确率对比

查看哪些样本在不同配置下答对/答错了。

In [ ]:
# 构建逐样本对比表
comparison_rows = []
for idx in range(LIMIT):
    row = {'idx': idx, 'ref': ref_answers[idx]}
    for res in all_configs:
        ans_info = res['answers'][idx]
        row[f"{res['config']}_ans"] = ans_info['gen_answer']
        row[f"{res['config']}_ok"] = 'Y' if ans_info['correct'] else 'N'
    comparison_rows.append(row)

df_comp = pd.DataFrame(comparison_rows)

# 找出配置间结果不一致的样本
diff_mask = (
    (df_comp['baseline_ok'] != df_comp['expand_no_rewarm_ok']) |
    (df_comp['baseline_ok'] != df_comp['expand_rewarm_ok'])
)
print(f"配置间结果不一致的样本: {diff_mask.sum()}/{LIMIT}")
print()

if diff_mask.sum() > 0:
    display(df_comp[diff_mask].head(20))
else:
    print("所有样本在三个配置下结果一致。")

## 13. 累计解码进度曲线

展示每个 config 在各步累计已解码 token 占比（即 1 - remaining/total_masks），反映解码速度。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for res in all_configs:
    cname = res['config']
    
    # 对每个样本，计算每步的累计已解码比例
    all_cum_ratios = defaultdict(list)
    for sample_records in res['all_step_records']:
        cum_transferred = 0
        for r in sample_records:
            cum_transferred += r['transferred']
            ratio = cum_transferred / GEN_LENGTH  # 占总生成长度的比例
            all_cum_ratios[r['global_step']].append(ratio)
    
    max_step = max(all_cum_ratios.keys()) if all_cum_ratios else 0
    steps_range = list(range(max_step + 1))
    means = [np.mean(all_cum_ratios[s]) if s in all_cum_ratios else 0 for s in steps_range]
    
    ax.plot(steps_range, means, label=config_labels[cname], color=colors[cname], linewidth=2)

ax.set_xlabel('Global Step', fontsize=12)
ax.set_ylabel('Cumulative Decoded Ratio', fontsize=12)
ax.set_title('Cumulative Decode Progress (GSM8K, 100 samples)', fontsize=14, fontweight='bold')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='100% decoded')
ax.legend(fontsize=11)
ax.set_xlim(0, 80)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../eval_results/cumulative_decode.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. 从 JSON 加载结果（跨机器汇总用）

如果是在多台机器/多个 GPU 上分别跑完的，把 JSON 文件拷过来后运行此 cell 即可重建所有图表。

In [ ]:
# === 从 JSON 重建结果（可选，跨机器汇总时取消注释） ===
# import glob
#
# json_files = sorted(glob.glob('../eval_results/gsm8k_*.json'))
# all_configs = []
# for jf in json_files:
#     with open(jf, 'r') as f:
#         data = json.load(f)
#     all_configs.append(data)
#     print(f"Loaded: {jf} -> {data['config']} Acc={data['accuracy']*100:.1f}%")
#
# # 重新设置 colors/labels
# colors = {'baseline': '#2196F3', 'expand_no_rewarm': '#FF9800', 'expand_rewarm': '#4CAF50'}
# config_labels = {'baseline': 'Baseline (dual cache)', 'expand_no_rewarm': 'Expand (no rewarm)', 'expand_rewarm': 'Expand (rewarm)'}
# GEN_LENGTH = all_configs[0]['gen_params']['gen_length']
#
# # 然后重新运行 Section 7-13 的 cell 即可生成所有图表
print("如需从 JSON 加载，取消上方注释后运行。")